In [1]:
import numpy as np
import pandas as pd


# ============================================================
# Load window features
# ============================================================

window_features = pd.read_parquet(
    "../data/processed/window_features.parquet"
)


# ============================================================
# Metadata columns
# ============================================================

metadata_columns = [
    "identifier",
    "window",
    "window_start",
    "window_end",
    "phase",
    "batch",
    "operating_point",
    "experiment_type",
    "experiment",
    "anomaly_label",
]


# ============================================================
# Model features
# ============================================================

model_features = [
    column
    for column in window_features.columns
    if column not in metadata_columns
]

print(f"Number of model features: {len(model_features)}")


# ============================================================
# Sort windows within each experiment
# ============================================================

window_features = (
    window_features
    .sort_values(["identifier", "window"])
    .reset_index(drop=True)
)


# ============================================================
# Previous-window features
# ============================================================

previous_features = (
    window_features
    .groupby("identifier", sort=False)[model_features]
    .shift(1)
)

previous_features.columns = [
    f"{column}_prev"
    for column in model_features
]


# ============================================================
# Delta features
# ============================================================

delta_values = (
    window_features[model_features].to_numpy()
    - previous_features.to_numpy()
)

delta_features = pd.DataFrame(
    delta_values,
    columns=[
        f"{column}_delta"
        for column in model_features
    ],
    index=window_features.index
)

# ============================================================
# Current + previous + delta features
# ============================================================

temporal_features = pd.concat(
    [
        window_features,
        previous_features,
        delta_features,
    ],
    axis=1
)


# ============================================================
# Validation
# ============================================================

print("\nTemporal feature matrix:")
print(temporal_features.shape)

print(
    f"Current features:  {len(model_features)}"
)
print(
    f"Previous features: {len(model_features)}"
)
print(
    f"Delta features:    {len(model_features)}"
)

print(
    f"\nExpected total columns: "
    f"{len(window_features.columns) + 2 * len(model_features)}"
)

print(
    f"Actual total columns:   "
    f"{len(temporal_features.columns)}"
)


# ============================================================
# Save to parquet
# ============================================================


temporal_features.to_parquet(
    "../data/processed/temporal_features.parquet",
    index=False
)

print(f"Data saved to parquet!")


# ============================================================
# Check first window of each experiment
# ============================================================

first_windows = (
    temporal_features
    .groupby("identifier")
    .head(1)
)

previous_nan_count = (
    first_windows[
        [f"{column}_prev" for column in model_features]
    ]
    .isna()
    .all(axis=1)
    .sum()
)

delta_nan_count = (
    first_windows[
        [f"{column}_delta" for column in model_features]
    ]
    .isna()
    .all(axis=1)
    .sum()
)

print(
    f"\nExperiments with no previous window: "
    f"{previous_nan_count}"
)

print(
    f"Experiments with no delta for first window: "
    f"{delta_nan_count}"
)

Number of model features: 240

Temporal feature matrix:
(11512, 730)
Current features:  240
Previous features: 240
Delta features:    240

Expected total columns: 730
Actual total columns:   730
Data saved to parquet!

Experiments with no previous window: 119
Experiments with no delta for first window: 119


In [2]:
temporal_features

,identifier,phase,batch,operating_point,experiment_type,experiment,window,window_start,window_end,anomaly_label,...,T703_T709_diff_std_delta,T703_T709_diff_min_delta,T703_T709_diff_max_delta,T704_T706_diff_slope_delta,T706_T708_diff_slope_delta,T704_T708_diff_slope_delta,T701_T702_diff_slope_delta,T702_T703_diff_slope_delta,T711_T712_diff_slope_delta,T703_T709_diff_slope_delta
0,Operation/batch_dist_binary_ethanol+propan-2-o...,Operation,batch_dist_binary_ethanol+propan-2-ol,operating_point_001,test_anormal,experiment_001,0,0,60,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Operation/batch_dist_binary_ethanol+propan-2-o...,Operation,batch_dist_binary_ethanol+propan-2-ol,operating_point_001,test_anormal,experiment_001,1,60,120,0.0,...,0.015965,-6.000000e-02,0.10,-0.000318,0.000486,0.000168,0.000379,0.001447,0.000127,-0.000435
2,Operation/batch_dist_binary_ethanol+propan-2-o...,Operation,batch_dist_binary_ethanol+propan-2-ol,operating_point_001,test_anormal,experiment_001,2,120,180,0.0,...,0.023577,-4.000000e-02,-0.08,-0.001368,-0.002504,-0.003871,-0.003972,-0.001482,0.000253,-0.002099
3,Operation/batch_dist_binary_ethanol+propan-2-o...,Operation,batch_dist_binary_ethanol+propan-2-ol,operating_point_001,test_anormal,experiment_001,3,180,240,0.0,...,-0.001568,3.000000e-02,0.02,0.003505,0.002483,0.005988,0.004347,0.002380,-0.000073,0.003565
4,Operation/batch_dist_binary_ethanol+propan-2-o...,Operation,batch_dist_binary_ethanol+propan-2-ol,operating_point_001,test_anormal,experiment_001,4,240,300,0.0,...,-0.009587,1.000000e-02,0.00,-0.001440,-0.001276,-0.002716,0.001114,0.000003,0.000034,0.000182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11507,Operation/batch_dist_ternary_butan-1-ol+propan...,Operation,batch_dist_ternary_butan-1-ol+propan-2-ol+water,operating_point_035,train_normal,experiment_001,66,3960,4020,0.0,...,0.001694,-2.000000e-01,-0.10,-0.010728,-0.002581,-0.013309,0.012223,0.000422,0.000392,0.007069
11508,Operation/batch_dist_ternary_butan-1-ol+propan...,Operation,batch_dist_ternary_butan-1-ol+propan-2-ol+water,operating_point_035,train_normal,experiment_001,67,4020,4080,0.0,...,-0.047464,1.000000e-01,-0.10,-0.015785,0.001628,-0.014157,-0.000772,-0.000217,0.000603,-0.003629
11509,Operation/batch_dist_ternary_butan-1-ol+propan...,Operation,batch_dist_ternary_butan-1-ol+propan-2-ol+water,operating_point_035,train_normal,experiment_001,68,4080,4140,0.0,...,0.016906,1.000000e-01,0.00,-0.016621,0.002478,-0.014143,-0.011584,0.000147,-0.000764,0.001564
11510,Operation/batch_dist_ternary_butan-1-ol+propan...,Operation,batch_dist_ternary_butan-1-ol+propan-2-ol+water,operating_point_035,train_normal,experiment_001,69,4140,4200,0.0,...,0.026693,0.000000e+00,0.10,-0.016880,0.004215,-0.012665,0.001423,0.000086,-0.000022,0.001698
